# ExcelNet

**ID** — Workbook `.xlsx`: sel, formula, style, format bersyarat, CSV/JSON, DataFrame.
**EN** — `.xlsx` workbooks: cells, formulas, styles, conditional formatting, CSV/JSON, DataFrames.

> Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil.

Panduan lengkap / full guide: [`docs/ExcelNet.md`](../docs/ExcelNet.md) ·
[Bahasa Indonesia](../docs/id/ExcelNet.md)

In [ ]:
// Build first:  dotnet build OfficeNet.sln -c Release

#r "../src/OfficeNet.Core/bin/Release/net10.0/Gravicode.OfficeNet.Core.dll"
#r "../src/PdfNet/bin/Release/net10.0/Gravicode.OfficeNet.PdfNet.dll"
#r "../src/ExcelNet/bin/Release/net10.0/Gravicode.OfficeNet.ExcelNet.dll"
#r "../src/OfficeNet.Rendering/bin/Release/net10.0/Gravicode.OfficeNet.Rendering.dll"

// Published instead? Swap the lines above for:
//   #r "nuget: Gravicode.OfficeNet, *"

In [ ]:
using OfficeNet.Rendering;
using Microsoft.DotNet.Interactive.Formatting;

// Renders a document and shows the first page inline, so a cell's effect is visible rather than
// described. Base64 in an <img> because the notebook has nowhere to serve a file from.
void Show(string path, int width = 520)
{
    var png = DocumentRenderer.RenderThumbnail(path, width);
    var data = Convert.ToBase64String(png);

    display(HTML($"<img src='data:image/png;base64,{data}' style='border:1px solid #ddd' />"));
}

var work = Path.Combine(Path.GetTempPath(), "officenet-notebook");
Directory.CreateDirectory(work);
string At(string name) => Path.Combine(work, name);

Console.WriteLine($"Berkas ditulis ke / files written to: {work}");

## Sel bertipe / Typed cells

`Set(object?)` memilih tipe sel dari tipe runtime. Tanggal adalah angka *ditambah* format angka —
memang hanya itu arti tanggal di spreadsheet. /
`Set(object?)` picks the cell type from the runtime type. A date is a number *plus* a number
format — that is all a date is in a spreadsheet.

In [ ]:
using ExcelNet;
using ExcelNet.Styles;
using OfficeNet.Core.Drawing;

var workbook = Workbook.Create("Penjualan");
var sheet = workbook["Penjualan"];

sheet.WriteHeader("A1", ["Tanggal", "Produk", "Wilayah", "Qty", "Harga", "Total"]);

string[] products = ["WordNet", "ExcelNet", "PowerPointNet", "PdfNet"];
string[] regions  = ["Jakarta", "Bandung", "Surabaya"];
var random = new Random(7);

for (var i = 0; i < 12; i++)
{
    var row = i + 1;
    sheet[row, 0].Set(new DateTime(2026, 1, 5).AddDays(i * 4));
    sheet[row, 1].Set(products[i % products.Length]);
    sheet[row, 2].Set(regions[i % regions.Length]);
    sheet[row, 3].Set(random.Next(4, 40));
    sheet[row, 4].Set(random.Next(60, 280) * 1000.0).WithNumberFormat(NumberFormats.Rupiah);
    sheet[row, 5].SetFormula($"D{row + 1}*E{row + 1}").WithNumberFormat(NumberFormats.Rupiah);
}

$"{sheet.RowCount} baris, {sheet.CellCount} sel terpakai"

## Formula — dan kenapa `Recalculate()` wajib / and why `Recalculate()` matters

Sel formula punya dua bagian: ekspresi dan hasil cache. Excel menghitung ulang saat dibuka; semua
konsumen lain membaca cache. Tanpa `Recalculate()`, PDF dan Google Sheets menampilkan nol. /
A formula cell has two parts: the expression and a cached result. Excel recomputes on open; every
other consumer reads the cache. Without `Recalculate()`, PDFs and Google Sheets show zeros.

In [ ]:
sheet["E14"].Set("TOTAL").Bold();
sheet["F14"].SetFormula("SUM(F2:F13)").WithStyle(
    CellStyle.Default.Bold()
        .WithNumberFormat(NumberFormats.Rupiah)
        .WithBackground(OfficeColor.FromRgb(0xFF, 0xF2, 0xCC)));

sheet["E15"].Set("RATA-RATA").Bold();
sheet["F15"].SetFormula("AVERAGE(F2:F13)").WithNumberFormat(NumberFormats.Rupiah);

Console.WriteLine($"Sebelum Recalculate: {sheet["F14"].Number:N0}");

workbook.Recalculate();

Console.WriteLine($"Sesudah  Recalculate: {sheet["F14"].Number:N0}");

Mesinnya meniru aritmetika Excel di tempat Excel berbeda dari .NET. /
The engine reproduces Excel's arithmetic where Excel differs from .NET.

In [ ]:
using var quirks = Workbook.Create("Quirks");
var q = quirks["Quirks"];

q["A1"].SetFormula("ROUND(2.5,0)");    // 3, bukan 2 (bukan pembulatan bankir)
q["A2"].SetFormula("MOD(-3,2)");       // 1, mengikuti tanda pembagi
q["A3"].SetFormula("-2^2");            // 4, minus uner mengikat lebih kuat
q["A4"].SetFormula("1/0");             // #DIV/0! sebagai nilai, bukan exception
q["A5"].SetFormula("IFERROR(A4,\"aman\")");

quirks.Recalculate();

foreach (var address in new[] { "A1", "A2", "A3", "A4", "A5" })
    Console.WriteLine($"{address}  {q[address].Formula,-22} = {q[address].Text}");

## Style dan format bersyarat / Styles and conditional formatting

`CellStyle` immutable — setiap `With…` mengembalikan yang baru. /
`CellStyle` is immutable — every `With…` returns a new one.

In [ ]:
var header = CellStyle.Default
    .Bold()
    .WithColor(OfficeColor.White)
    .WithBackground(OfficeColor.FromRgb(0x1F, 0x38, 0x64))
    .WithAlignment(HorizontalAlignment.Center);

sheet.Range("A1:F1").ApplyStyle(header);

sheet.AddColorScale("D2:D13",
    OfficeColor.FromRgb(0xF8, 0x69, 0x6B),
    OfficeColor.FromRgb(0x63, 0xBE, 0x7B));

sheet.Frozen = new FreezePanes(Rows: 1, Columns: 0);
sheet.AutoFitColumns();
sheet.TabColor = OfficeColor.FromRgb(0x1F, 0x38, 0x64);

workbook.Save(At("excelnet.xlsx"));
Show(At("excelnet.xlsx"), 700);

## CSV dan JSON

`CsvOptions.Indonesian` penting: locale yang memakai `,` sebagai desimal harus memakai `;` sebagai
pemisah kolom. Salah di sini mengubah tiap angka jadi dua kolom. /
`CsvOptions.Indonesian` matters: a locale using `,` as the decimal separator must use `;` as the
field separator. Getting it wrong turns every number into two columns.

In [ ]:
using ExcelNet.Io;

var csv = CsvIo.ExportText(sheet, CsvOptions.Indonesian);
Console.WriteLine(string.Join("\n", csv.Split('\n').Take(4)));

Console.WriteLine();
var json = JsonIo.ExportText(sheet);
Console.WriteLine(json[..Math.Min(260, json.Length)] + "…");

## Ekspor PDF / PDF export

In [ ]:
workbook.SaveAsPdf(At("excelnet.pdf"), new ExcelPdfOptions
{
    ShowGridLines = true,
    RepeatHeaderRow = true,
});

Show(At("excelnet.pdf"), 700);

In [ ]:
workbook.Dispose();